# Decision Appendix: `03_forensic_clean_for_analysis`

**Author:** Amy Zhang  
**AI Collaborators:** ChatGPT, Claude Code, Perplexity AI  
**Date:** January - February 2026

---

# Prelude 

## Re-entry Points

- **01_data_ingestion.ipynb**  
  Constructed the “ground truth” dataset: fully merged, multi-year thermoelectric cooling data.

- **02_profile_analysis.ipynb**  
  Identified the effective observation grain of the dataset and surfaced alternative “vantage-point” grains implicit in the data structure.

- ➡️ **03_forensic_clean_for_analysis.ipynb (this notebook)**  
  The goal of this notebook is to **forensically explore the available metrics for analysis**, with particular focus on **water use metrics**.  


## Next Steps

1. **Center the analysis on water volume, intensity, and per-fuel consumption metrics.**  
   If we accept these metrics as the most complete signals in the dataset, what grain do they imply?

2. **Only after establishing the water-metrics grain should sub-questions and imputation be explored.**  
   Without this step, there is a risk of imputing values for configurations where water metrics should not exist at all.

   Any imputation strategy must account for:
   - Persisting water metrics alongside non-operable equipment states
   - Temporal misalignment between observation dates and in-service dates
   - The structural complexity of generator–boiler–cooling relationships

---

# 1. Import Libraries & Dataset  

---
# 2. Water Metrics: Validation of Formula Reconstruction for `intensity` and `rate_per_fuel_consumption` Columns (Conclusion: Correct)

- At the end of `02_profile_analysis.ipynb`, we derived our own withdrawal and consumption intensity and rate_per_fuel_consumption columns.
- While the mean absolute difference showed our intensity columns to match the logic of the original columns, rate_per_fuel_consumption varied dramatically.

>```
>=== withdrawal_intensity_gallons_mwh ===
>  Rows with both: 57450
>  Mean abs diff: 0.25
>  Exact matches (diff < 0.01): 1478
>
>=== consumption_intensity_gallons_mwh ===
>  Rows with both: 51851
>  Mean abs diff: 0.17
>  Exact matches (diff < 0.01): 17321
>
>=== withdrawal_per_fuel_gallons_mmbtu ===
>  Rows with both: 53945
>  Mean abs diff: 96110.78
>  Exact matches (diff < 0.01): 2046
>
>=== consumption_per_fuel_gallons_mmbtu ===
>  Rows with both: 48420
>  Mean abs diff: 318.26
>  Exact matches (diff < 0.01): 15674
>```

- We resume by exploring withdrawal_per_fuel_gallons_mmbtu.

##   2a) `derived_withdrawal_per_fuel_gallons_mmbtu` (new) v. `water_withdrawal_rate_per_fuel_consumption_gallons__mmbtu` (original)    

#### Notes: 

- Up to the 95th percentile, the absolute difference stay below half a gallon per MMBtu, meaning the two columns have high agreement. 
- The tail is where the absolute difference explodes: at 99% the difference becomes 23.8 gallons per MMBtu, and by 100%, 4.6 billion.
- A tiny number of rows heavily skewed the mean absolute difference uncovered earlier.

#### Next step: 

- Examine these rows.

#### Conclusion
- Absolute differences can be misleading when values vary widely in scale. Examining **relative differences** confirms that our formula for reconstructing **withdrawal rate per fuel** is robust for nearly all rows.  
- Instances of **infinite relative differences** occur when the reported water withdrawal rate per fuel consumption was rounded to **0.0**, not because of formula errors.  
- We will now apply the same validation process to **consumption rate per fuel** to ensure consistency.

##   2b) `derived_consumption_per_fuel_gallons_mmbtu` (new) v. `water_consumption_rate_per_fuel_consumption_gallons__mmbtu` (original) 

#### Conclusion
- The mean absolute difference is inflated by a tiny number of extreme observations.  
- The top 5% of relative differences occur mainly because the reported values were originally zero, similar to what we observed for withdrawal rates.  
- Overall, our reconstructed formulas are robust.
---
# 3. Exploring Reporting Inconsistencies    
##   3a) Water Consumption-Only Rows (Conclusion: Plausible)  

#### Domain Question: 
- Is it plausible to have withdrawal without consumption or vice-versa?

#### ChatGPT: 

- Is it physically plausible to have water withdrawal without any consumption?
    - Yes, in some cooling systems (like once-through or recirculating systems with mostly return flow), you might withdraw water but not consume it.
- Is it plausible to have consumption without withdrawal?
    - Less likely, since “consumption” usually comes from the withdrawn water being lost (evaporation, blowdown, process use). That 0.4% could be tiny rounding errors, reporting quirks, or special cases.

#### 1. Create flags & explore in relation to paradox timeline flag.
> ```
> any_paradox_flag   False  True 
> water_metric_flag              
> both               62.39   8.30
> consumption_only    0.41   0.16
> neither            32.56  91.42
> withdrawal_only     4.64   0.12
> ```
- The vast majority of paradox rows (~91%) fall into the neither category, meaning they report neither water withdrawal nor water consumption volumes. This suggests that missing water metrics in paradox rows are primarily structural, not the result of random data quality issues.

#### 2. Exploring `consumption_only` rows
These rows may represent cooling systems where evaporative losses are reported but withdrawals are structurally missing or rounded.
- Are consumption_only rows fundamentally different in magnitude/shape, or are they “normal” consumption observations missing withdrawal for structural reasons?

|        stat        | consumption_only_log |   both_log   |
|--------------------|----------------------|--------------|
| count              | 2834.000000          | 445810.000000|
| mean               | 1.760322             | 1.463510     |
| std                | 0.460693             | 0.939751     |
| min                | -0.723538            | -3.000000    |
| 25%                | 1.656354             | 1.110792     |
| 50%                | 1.792462             | 1.632660     |
| 75%                | 1.878522             | 1.964585     |
| max                | 3.967447             | 5.099432     |


#### ChatGPT:
This pattern strongly supports:
- consumption_only rows represent real cooling-related water losses occurring at configurations where withdrawals are not reported at this grain.

**Very plausible explanations:**
- cooling systems drawing from recirculated sources
- withdrawal measured elsewhere (plant-level, not cooling-unit-level)
- regulatory/reporting asymmetry (consumption emphasized over intake)

### Conclusion: `consumption_only` rows are valid measurements
- Missing withdrawal metrics are structural: the majority of consumption-only rows are measurements from closed/recirculated water systems.
- Most water comes from reclaimed or previously discharged sources, so imputation of withdrawal is unnecessary.

##   3b) Negative Value Rows (Decision: Remove)  

#### **Notes** (Claude AI-assisted):

- Plant 620:
    - Large plant (~ 20k MG withdrawal)
    - The negatives are only in consumption and only in 2014-2015.
    - Tiny relative to scale, likely early reporting errors. Low concern.

- Plant 2393
    - Clear operational decline.
    - High activity 2014-2016, then steadily drops toward near-zero after 2018.
    - Negatives are tiny in absolute terms.
    - This plant may have scaled down or retired units — the negatives are almost noise here.

- Plant 6043:
    - A wild one: dramatic regime change ~ 2016-2017: withdrawal goes from wildly volatile (up to 50k MG) to suddenly flat and stable (~11k MG).
    - The consumption negatives are clustered in that chaotic early period.
    - Something major happened — technology switch, cooling system change, unit retirement.
    - The negatives are probably a symptom of the instability before whatever changed.

- Plant 6081:
    - Erratic withdrawals throughout
    - Near-zero consumption most of the time, then that one enormous consumption spike ~2021 that looks like a data entry error more than anything real.

#### Next Steps
- The negatives across all four feel like byproducts of specific operational contexts rather than systematic reporting errors — these are individual "plant stories."
- Crucially, these plots are at the raw equipment grain (generator/boiler/cooling unit per month), not aggregated to plant-month. The gaps in the consumption line reflect NaN values — months where consumption wasn't reported for a given unit.
- Open question: do the negative rows largely disappear once aggregated to plant-month?

#### Decision: Remove Negative Water Metric Rows

Negative values for `water_withdrawal_volume_million_gallons` and `water_consumption_volume_million_gallons` persist even after aggregating to the plant-month level, ruling out the possibility that they are artifacts of the raw equipment grain. Physically, neither metric can be negative — a plant cannot "produce" water back into its source.

**Potential bias:** Removal is concentrated in a small number of plants during specific periods (notably plants 620, 2393, 6043, and 6081 in the 2014–2016 window). If these negative entries represent reporting corrections, their removal may slightly overstate water use for those plants in those years. However, at 1,070 rows out of ~ 917,000 (~ 0.12%), the dataset-level impact is negligible.

**Decision:** Remove all rows where either water metric is negative. The isolated, plant-specific pattern of these anomalies suggests data entry or reporting error, or symptoms of plant-level issues that manifest through other features (the overall shape of withdrawal relative to consumption, for instance). Removal poses minimal risk of overgeneralizing or smoothing out water usage patterns; that risk would be greater with imputation.
##   3c) Neither Water Metric  

First, we generated a summary of categorical columns to see whether rows without either water metric capture a consistent condition. Second, we did a rate comparison for the attributes where a value represented over 50% of the 'neither water metrics' subset with the attributes' distribution in the subset of all other rows in the dataset. 

#### Results (with Claude AI assistance re: domain): 

1. NaN cooling_type_923 (67.5% vs 0.0%) — Cooling_type_923 comes from EIA-923 Schedule 8, which is specifically the cooling water supplement. If a plant doesn't file Schedule 8, they get NaN for cooling type and no water metrics — they're absent from the same form. These aren't missing values, they're structural non-reporters.

2. DC (dry cooling) (14.9% vs 0.1%) — Dry-cooled plants use air instead of water, so they genuinely have no water to report. Of course they have neither metric — they're not cooling with water at all. Zero is the correct answer, not missing data.

3. Together these two categories account for ~ 82.4% of neither rows, and both have completely legitimate explanations. The remaining ~17.6% of neither rows (cooling types ON, RI, RC etc.) lack an obvious structural explanation and may warrant further investigation, though their share is small.

4. Consistent with findings 1 and 2: steam turbines and bituminous coal are over-represented in neither rows, reflecting older coal-fired steam plants that are more likely to use dry cooling or fall outside EIA-923 Schedule 8 reporting requirements. Conversely, natural gas (NG) and combined-cycle turbines are under-represented — these plants tend to be newer and more consistently captured in Schedule 8.

#### Next Step: 

- Remove structurally explained rows from melted_df: `water_metric_flag` == `neither` and `cooling_type_923` is `NaN` or `DC`

###       3c1) Remove Structurally Explained Rows (`cooling_type_923` is `NaN` or `DC`)  
###       3c2) Explore Remaining Rows with Missing Water Metrics: Plant-Level Investigation  
- 387 plants still have rows with neither water metric reported.
- Neither rows are concentrated (top 5 plants account for ~39% of the 53,984 rows)
- To investigate whether missingness follows a temporal pattern, we normalize each plant-year's neither count by the plant's total observations for that year and visualize the top 20 plants by neither row count as a heatmap.

### Notes for Heatmap: 

- Visualizing the distribution of rows with neither water metric across each year for the top 20 plants, the following patterns emerge:
1) Plants that seems to 'turn off': all their observations in a year have data up until a certain year, after which both water metrics go absent. ex: 10360 ('turns off' in 2018); 50189 ('turns off' in 2021); 59773 ('turns off' in 2016)
2) Plants that seem to 'turn on': all these observations are missing water metrics until a certain year (50707, 54676)
3) Partial missingness distributed across all years (ex: 2322)
4) One year of no missingness or less, surrounded by 100% missing (54101 only has water metric data for 2015; 613 seems to come partially back 'online' in 2022, before 'turning back off')

#### 1. Hypothesis: SCD Type 2 Dimensions
- To test the hypothesis that the changes in water metric data availability are symptoms of a slowly changing dimension attribute at the plant level, we will inspect the following for four plants from the heatmap:
  - unique `technology_type` values per unique `water_metric_flag` per year
  - the same for `generator_primary_energy_source_code`, `cooling_system_type`, `sector`, `relationship_type`, `utility_id`, `generator_status`, `boiler_status`, and `cooling_status`
- If there is a change in any of these that coincides with the change in `water_metric_flag`, this suggests the attribute is stable at the plant level at any given point in time, but changes over the dataset's time period — i.e., SCD Type 2 behavior.

#### **Result:** 
- No clean SCD pattern was found. For most plants, categorical
    attributes remain stable across the turn-on/turn-off year. The exception is
    plant 2322, where `both` and `neither` rows coexist in every year with
    identical attributes — suggesting sub-plant grain missingness rather than
    a dimension change. 
    
- Additionally, the missing years in some plants reveal a heatmap artifact:                          
  years with no observations at all also display as 0% neither,
  indistinguishable from years with complete reporting. (Notice 2015 missing from Plant 54104)

#### 2. Hypothesis: Fuel-Type and Sector-Related Missingness

#### Claude AI - Fuel-Type and Sector-Related Missingness Hypothesis: 

> "Why CHP specifically:                                                                              
  Combined Heat and Power plants capture waste heat for industrial processes, which means their
  cooling needs are fundamentally different from a conventional power plant. A conventional steam    
  plant needs to reject enormous amounts of heat through cooling water. A CHP plant uses that heat
  productively, so the cooling load is much smaller or sometimes negligible. Less cooling water
  needed → less to report → potentially below EIA reporting thresholds.

> Why Industrial CHP specifically (vs IPP CHP):
  Industrial CHP is embedded in a manufacturing process — paper mill, chemical plant, refinery. The
  "cooling" may happen through the industrial process itself rather than a dedicated cooling system.
  IPP CHP (independent power producers) are more standalone and more likely to fall under standard
  utility reporting requirements.

> Why BLQ, BIT, WDS:
  These are all fuels associated with industrial facilities — black liquor (paper mills), wood/wood
  waste (lumber, biomass facilities), bituminous coal in industrial settings. These plants exist
  primarily to serve an industrial process, with electricity as a byproduct. The power generation is
  almost incidental, which likely affects their EIA reporting obligations."

#### Notes: 
- We examined the distribution of various attributes for plant 2322, comparing
  rows with neither water metric against rows with both water metrics. The results
  show an even split across all attributes at the cooling unit grain (`generator_id`,
  `boiler_id`, `cooling_id`), providing no clear signal for why reporting toggles
  within this plant.
- We next examine a hypothesis (surfaced through Claude AI) that some missingness
  is explained by sector and fuel type — specifically whether Industrial CHP
  and certain fuel types are systematically over-represented in `neither_df_2`.

###       3c3) Conclusion: Remove Rows with Neither Water Metric
  - While Industrial CHP is over-represented among rows with neither metric, and
    its associated fuels are also over-represented, sectors where water metric
    reporting would be expected (e.g. Electric Utility, IPP Non-CHP) are also
    significantly present — the pattern is not clean enough to remove by sector
    or fuel alone.
  - As surfaced elsewhere in these Phase 1 notebooks, the EIA thermoelectric
    cooling dataset contains multidimensional complexity across ~ 70 attributes,
    where a plant's story emerges at the individual equipment level (see:
    `decision_appendix.ipynb`: '`02_profile_analysis.ipynb`: 4b) Paradox Timelines).
  - After removing structurally explained rows (`cooling_type_923` is `NaN` or
    `DC`), remaining neither rows represent 53,984 out of ~ 662K rows.
  - As these rows likely reflect genuinely absent water usage or sub-threshold
    reporting rather than missing data, imputation would risk inflating water
    usage findings. We therefore remove them from the analysis dataset.
---
# 4. Save `analysis_ready_df.csv`  